# Advanced Pandas Tutorial for Data Professionals

This notebook covers **essential advanced topics** beyond the basics:
- MultiIndex / Hierarchical Indexing
- Reshaping Data (`melt`, `pivot`, `explode`)
- Advanced Time Series (`resample`, `rolling`, `shift`, `diff`, `expanding`, `ewm`)
- Advanced GroupBy (`transform`, `filter`, named aggregation, `pd.Grouper`)
- Advanced Merging (`indicator`, index joins, `merge_asof`, `merge_ordered`)
- Vectorized String Methods with Regex
- Categorical Data
- Advanced Missing Data Handling (interpolation, group‑based fill)
- Performance Optimization (`query`, `eval`, downcasting, chunking)
- Working with Databases (`read_sql`, `to_sql`)
- JSON Normalization
- Memory Profiling

All examples are self‑contained and runnable.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# For SQL examples
import sqlite3
from sqlalchemy import create_engine

np.random.seed(42)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

## 1. MultiIndex / Hierarchical Indexing

MultiIndex allows you to work with higher‑dimensional data in a 2D DataFrame.

In [ ]:
# Create a MultiIndex DataFrame
arrays = [
    ['A', 'A', 'B', 'B', 'C', 'C'],
    ['X', 'Y', 'X', 'Y', 'X', 'Y']
]
index = pd.MultiIndex.from_arrays(arrays, names=['letter', 'sub'])
df_multi = pd.DataFrame({'value': [10, 20, 30, 40, 50, 60]}, index=index)
print("MultiIndex DataFrame:")
print(df_multi)

# Access with .loc using tuple
print("\n.loc[('A', 'X')]:", df_multi.loc[('A', 'X')])

# Cross-section using .xs
print("\n.xs('X', level='sub'):")
print(df_multi.xs('X', level='sub'))

# Stack / Unstack
df_unstacked = df_multi.unstack(level='sub')
print("\nUnstacked (sub → columns):\n", df_unstacked)
df_stacked = df_unstacked.stack()
print("\nStacked back:\n", df_stacked)

## 2. Reshaping Data: `melt`, `pivot`, `explode`, `crosstab`

Convert between wide and long formats – critical for plotting and modelling.

In [ ]:
# Sample wide data
df_wide = pd.DataFrame({
    'Name': ['Alice', 'Bob'],
    'Math': [90, 85],
    'Physics': [88, 92],
    'Chemistry': [95, 80]
})
print("Wide format:")
print(df_wide)

# Melt (wide → long)
df_long = pd.melt(df_wide, id_vars=['Name'], var_name='Subject', value_name='Score')
print("\nLong format (melted):")
print(df_long)

# Pivot (long → wide)
df_wide_back = df_long.pivot(index='Name', columns='Subject', values='Score')
print("\nPivoted back to wide:")
print(df_wide_back)

# pivot_table (with aggregation, e.g., for duplicate entries)
df_dup = pd.DataFrame({
    'Date': ['2024-01-01', '2024-01-01', '2024-01-02'],
    'Product': ['A', 'B', 'A'],
    'Sales': [100, 150, 200]
})
pivot_tab = pd.pivot_table(df_dup, values='Sales', index='Date', columns='Product', aggfunc='sum', fill_value=0)
print("\npivot_table with aggregation:")
print(pivot_tab)

# explode – turn list entries into rows
df_list = pd.DataFrame({'ID': [1, 2], 'Values': [[10, 20], [30, 40, 50]]})
print("\nBefore explode:")
print(df_list)
df_exploded = df_list.explode('Values')
print("\nAfter explode:")
print(df_exploded)

## 3. Advanced Time Series

Essential for finance, IoT, and forecasting.

In [ ]:
# Create daily data
dates = pd.date_range('2024-01-01', periods=100, freq='D')
ts = pd.Series(np.random.randn(100).cumsum(), index=dates, name='price')
df_ts = ts.to_frame()
print("Time series head:")
print(df_ts.head())

# Resample to weekly mean
weekly = df_ts.resample('W').mean()
print("\nWeekly resampled (mean):")
print(weekly.head())

# Rolling window (7-day moving average)
df_ts['MA7'] = df_ts['price'].rolling(window=7).mean()
print("\n7-day moving average (last 5 rows):")
print(df_ts.tail())

# Shift (lag) and diff (change)
df_ts['lag1'] = df_ts['price'].shift(1)
df_ts['diff1'] = df_ts['price'].diff()
print("\nShift and diff (first 5 rows):")
print(df_ts.head())

# Expanding window (cumulative mean)
df_ts['cummean'] = df_ts['price'].expanding().mean()
print("\nExpanding mean (last 5 rows):")
print(df_ts.tail())

# Exponential weighted moving average (EWM)
df_ts['ewm12'] = df_ts['price'].ewm(span=12, adjust=False).mean()
print("\nEWM (span=12) last 5 rows:")
print(df_ts[['price', 'ewm12']].tail())

## 4. Advanced GroupBy Operations

Beyond simple aggregation: `transform`, `filter`, named aggregation, time‑based grouping.

In [ ]:
# Sample data
df_sales = pd.DataFrame({
    'Region': ['North', 'North', 'South', 'South', 'East', 'East'],
    'Product': ['A', 'B', 'A', 'B', 'A', 'B'],
    'Sales': [100, 150, 200, 250, 300, 350],
    'Target': [120, 130, 180, 240, 280, 360]
})
print("Sales data:")
print(df_sales)

# Transform: fill missing values with group mean
df_sales_missing = df_sales.copy()
df_sales_missing.loc[0, 'Sales'] = np.nan
df_sales_missing['Sales_filled'] = df_sales_missing.groupby('Region')['Sales'].transform(lambda x: x.fillna(x.mean()))
print("\nTransform (fill NA with group mean):")
print(df_sales_missing)

# Filter: keep groups where total Sales > 400
filtered = df_sales.groupby('Region').filter(lambda g: g['Sales'].sum() > 400)
print("\nFilter groups with Sales sum > 400:")
print(filtered)

# Named aggregation
agg_named = df_sales.groupby('Region').agg(
    total_sales=('Sales', 'sum'),
    avg_sales=('Sales', 'mean'),
    total_target=('Target', 'sum')
)
print("\nNamed aggregation:")
print(agg_named)

# pd.Grouper for time-based grouping
dates = pd.date_range('2024-01-01', periods=30, freq='D')
df_time = pd.DataFrame({'date': dates, 'value': np.random.randn(30)})
grouped_weekly = df_time.groupby(pd.Grouper(key='date', freq='W')).sum()
print("\nWeekly sum using pd.Grouper:")
print(grouped_weekly.head())

## 5. Advanced Merging

Real‑world data comes from multiple tables. Master these join techniques.

In [ ]:
# Sample data
left = pd.DataFrame({'key': ['A', 'B', 'C', 'D'], 'value_left': [1, 2, 3, 4]})
right = pd.DataFrame({'key': ['B', 'C', 'E', 'F'], 'value_right': [5, 6, 7, 8]})
print("Left:\n", left)
print("\nRight:\n", right)

# Merge with indicator
merged_indicator = pd.merge(left, right, on='key', how='outer', indicator=True)
print("\nMerge with indicator (shows source):")
print(merged_indicator)

# Merge on index (join method)
left_idx = left.set_index('key')
right_idx = right.set_index('key')
joined = left_idx.join(right_idx, how='inner')
print("\nJoin on index (inner):\n", joined)

# merge_asof (nearest key, typically time series)
left_ts = pd.DataFrame({'time': pd.to_datetime(['2024-01-01', '2024-01-03', '2024-01-05']), 'price': [100, 102, 101]})
right_ts = pd.DataFrame({'time': pd.to_datetime(['2024-01-02', '2024-01-04']), 'volume': [1000, 1500]})
asof_merged = pd.merge_asof(left_ts, right_ts, on='time', direction='backward')
print("\nmerge_asof (backward):\n", asof_merged)

# merge_ordered (with interpolation)
ordered = pd.merge_ordered(left_ts, right_ts, on='time', fill_method='ffill')
print("\nmerge_ordered with forward fill:\n", ordered)

## 6. Vectorized String Methods with Regex

Clean and extract information from text columns.

In [ ]:
# Sample text data
df_text = pd.DataFrame({'name': ['John Smith', 'Jane Doe', 'Bob Johnson', 'Alice Brown'],
                        'email': ['john@example.com', 'jane.doe@work.org', 'bob@gmail.com', 'alice@company.co.uk'],
                        'phone': ['(123) 456-7890', '987-654-3210', '555-123-4567', '444-555-6666']})
print("Original:")
print(df_text)

# Extract domain from email using regex
df_text['email_domain'] = df_text['email'].str.extract(r'@([\w\.]+)')
print("\nExtracted email domain:")
print(df_text[['email', 'email_domain']])

# Extract first and last name from 'name' column
df_text[['first', 'last']] = df_text['name'].str.extract(r'(\w+)\s+(\w+)')
print("\nExtracted first & last name:")
print(df_text[['name', 'first', 'last']])

# Find all matches (extractall) – returns MultiIndex
df_numbers = pd.DataFrame({'text': ['abc 123 def 456', 'no numbers', '78 90']})
all_numbers = df_numbers['text'].str.extractall(r'(\d+)')
print("\nExtractall (all numbers):\n", all_numbers)

# Split and expand
split_names = df_text['name'].str.split(expand=True)
split_names.columns = ['first_split', 'last_split']
print("\nSplit name with expand=True:")
print(split_names)

# String contains for filtering
gmail_users = df_text[df_text['email'].str.contains('gmail', case=False)]
print("\nUsers with Gmail:\n", gmail_users)

## 7. Categorical Data

Save memory and enable ordered categories.

In [ ]:
# Convert to category
df_cat = pd.DataFrame({'size': ['S', 'M', 'L', 'XL', 'S', 'M']})
df_cat['size'] = df_cat['size'].astype('category')
print("Categorical column:\n", df_cat['size'])
print(f"Memory usage: {df_cat['size'].memory_usage(deep=True)} bytes")

# Define ordered categories
size_order = pd.CategoricalDtype(categories=['S', 'M', 'L', 'XL'], ordered=True)
df_cat['size_ordered'] = df_cat['size'].astype(size_order)
print("\nSorted by ordered categories:")
print(df_cat.sort_values('size_ordered'))

# Access category codes
df_cat['code'] = df_cat['size_ordered'].cat.codes
print("\nCategory codes:\n", df_cat)

## 8. Advanced Missing Data Handling

Interpolation, limit parameters, group‑based imputation.

In [ ]:
# Create data with missing values
df_missing = pd.DataFrame({
    'day': range(1, 11),
    'value': [10, np.nan, 12, np.nan, 14, 15, np.nan, 17, 18, 20]
})
print("Original missing data:")
print(df_missing)

# Linear interpolation
df_missing['linear'] = df_missing['value'].interpolate(method='linear')
print("\nLinear interpolation:")
print(df_missing)

# Forward fill with limit (only fill 1 missing forward)
df_missing['ffill_limit1'] = df_missing['value'].fillna(method='ffill', limit=1)
print("\nForward fill with limit=1:")
print(df_missing[['value', 'ffill_limit1']])

# Group‑based fill (fill missing with group mean)
df_group = pd.DataFrame({
    'group': ['A', 'A', 'B', 'B', 'A', 'B'],
    'value': [100, np.nan, 200, np.nan, 110, 190]
})
df_group['value_filled'] = df_group.groupby('group')['value'].transform(lambda x: x.fillna(x.mean()))
print("\nGroup‑based fill with group mean:")
print(df_group)

## 9. Performance Optimization

Speed up operations and reduce memory usage.

In [ ]:
# Use query() for faster filtering
df_large = pd.DataFrame({'x': np.random.randn(1000000), 'y': np.random.randn(1000000)})
%timeit df_large[(df_large['x'] > 0) & (df_large['y'] < 0)]
%timeit df_large.query('x > 0 and y < 0')

# Use eval() for arithmetic expressions
%timeit df_large['x'] + df_large['y']
%timeit df_large.eval('x + y')

# Downcast numeric columns to save memory
def downcast(df):
    for col in df.select_dtypes(include=['int', 'float']).columns:
        if df[col].dtype == 'int64':
            df[col] = pd.to_numeric(df[col], downcast='integer')
        elif df[col].dtype == 'float64':
            df[col] = pd.to_numeric(df[col], downcast='float')
    return df

df_large_downcast = downcast(df_large.copy())
print(f"Original memory: {df_large.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"Downcast memory: {df_large_downcast.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Read large file in chunks (example)
# for chunk in pd.read_csv('large.csv', chunksize=10000):
#     process(chunk)

## 10. Working with Databases (SQL)

Read from and write to SQL databases using `sqlalchemy`.

In [ ]:
# Create an in‑memory SQLite database
engine = create_engine('sqlite:///:memory:')

# Write DataFrame to SQL table
df_sql = pd.DataFrame({'id': [1, 2, 3], 'name': ['Alice', 'Bob', 'Charlie'], 'score': [85, 92, 78]})
df_sql.to_sql('students', engine, index=False, if_exists='replace')
print("DataFrame written to SQL table 'students'")

# Read entire table
df_read = pd.read_sql('SELECT * FROM students', engine)
print("\nRead entire table:")
print(df_read)

# Read with SQL query
query = "SELECT name, score FROM students WHERE score > 80"
df_filtered = pd.read_sql(query, engine)
print("\nQuery result (score > 80):")
print(df_filtered)

# Chunked reading
for chunk in pd.read_sql('SELECT * FROM students', engine, chunksize=2):
    print("Chunk:", chunk.to_dict(orient='records'))

## 11. JSON Normalization

Flatten nested JSON data into a DataFrame.

In [ ]:
# Sample nested JSON
nested_json = [
    {
        "id": 1,
        "name": "Alice",
        "address": {
            "city": "New York",
            "zip": "10001"
        },
        "orders": [{"product": "Laptop", "price": 1200}, {"product": "Mouse", "price": 25}]
    },
    {
        "id": 2,
        "name": "Bob",
        "address": {
            "city": "Boston",
            "zip": "02101"
        },
        "orders": [{"product": "Keyboard", "price": 80}]
    }
]

# Flatten with record_path and meta
df_orders = pd.json_normalize(
    nested_json,
    record_path='orders',
    meta=['id', 'name', ['address', 'city'], ['address', 'zip']]
)
print("Normalized orders:")
print(df_orders)

# If no record_path, flatten top level only
df_customers = pd.json_normalize(nested_json)
print("\nCustomers (top level):")
print(df_customers)

## 12. Memory Profiling

Understand and reduce memory footprint.

In [ ]:
# Check memory usage
df_sample = pd.DataFrame({
    'int64_col': np.random.randint(0, 100, size=10000),
    'float64_col': np.random.randn(10000),
    'object_col': ['abcdefghijklmnopqrstuvwxyz'] * 10000,
    'category_col': pd.Series(['A','B','C','D']*2500).astype('category')
})

print("Memory usage (bytes):")
print(df_sample.memory_usage(deep=True))

print("\nTotal memory:", df_sample.memory_usage(deep=True).sum() / 1024**2, "MB")

# Optimize object columns to category if cardinality low
cardinality = df_sample['object_col'].nunique()
if cardinality / len(df_sample) < 0.5:
    df_sample['object_col'] = df_sample['object_col'].astype('category')

print("\nAfter converting low‑cardinality object to category:")
print(df_sample.memory_usage(deep=True))

## Summary

You have now explored advanced Pandas topics essential for data professionals:
- **MultiIndex** – hierarchical indexing
- **Reshaping** – `melt`, `pivot`, `explode`
- **Time Series** – `resample`, `rolling`, `shift`, `diff`, `expanding`, `ewm`
- **GroupBy** – `transform`, `filter`, named aggregation, `pd.Grouper`
- **Merging** – indicator, index joins, `merge_asof`, `merge_ordered`
- **String methods** – regex extraction, splitting
- **Categorical** – memory savings, ordered categories
- **Missing data** – interpolation, limit, group‑based fill
- **Performance** – `query`, `eval`, downcasting, chunking
- **Databases** – `read_sql`, `to_sql`
- **JSON** – `json_normalize`
- **Memory profiling** – `memory_usage`, optimisation strategies

These skills will enable you to work efficiently with real‑world, messy, large‑scale data.